# Notebook 04 — L'ecosistema LLM nel 2026

**Modulo:** 04
**Tempo stimato:** 1,5-2 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Due funzioni minime, una per Claude (Anthropic) e una per GPT (OpenAI),
  usate per confrontare lo stesso prompt su provider diversi
- Un confronto diretto sul caso pratico "piano alimentare" della dispensa
- Un confronto tra risposta standard e risposta con ragionamento forzato
  (Chain of Thought), sugli stessi due provider
- Una piccola tabella di osservazioni compilata da te

> **Nota:** qui usiamo funzioni semplici e separate per provider, non un
> wrapper unificato. Il wrapper multi-provider "vero" lo costruiremo nel
> Modulo 08, quando avremo più esperienza con le differenze tra le API.

## Prerequisiti

```bash
pip install anthropic openai python-dotenv
```

Serve il `.env` con `ANTHROPIC_API_KEY` e `OPENAI_API_KEY` (lo stesso dei
moduli precedenti). Se hai anche `GOOGLE_API_KEY` e vuoi provare Gemini,
c'è un esercizio facoltativo alla fine.

---
## Cella 0 — Setup

In [ ]:
import os
from dotenv import load_dotenv
import anthropic
from openai import OpenAI

load_dotenv()
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")
OPENAI_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_KEY = os.getenv("GOOGLE_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    if chiave:
        print(f"OK  {nome}: {chiave[:8]}...")
    else:
        print(f"MANCA  {nome}: non trovata (necessaria solo se usi quel provider)")

verifica_chiave("ANTHROPIC_API_KEY", ANTHROPIC_KEY)
verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)
verifica_chiave("GOOGLE_API_KEY", GOOGLE_KEY)

cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
cliente_openai = OpenAI(api_key=OPENAI_KEY)


---
## Cella 1 — Due funzioni minime, un provider ciascuna

Niente wrapper unificato per ora: solo due funzioni semplici, una per
Claude e una per GPT, che restituiscono testo e token usati. È lo stesso
schema visto nel Modulo 01, solo un po' più ordinato.

In [ ]:
def chiama_claude(prompt: str, modello: str = "claude-sonnet-5-20260115", max_tokens: int = 700) -> dict:
    # verifica l'ID modello aggiornato nella doc ufficiale
    risposta = cliente_anthropic.messages.create(
        model=modello,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return {
        "provider": "anthropic",
        "modello": modello,
        "testo": risposta.content[0].text,
        "token_input": risposta.usage.input_tokens,
        "token_output": risposta.usage.output_tokens,
    }

def chiama_gpt(prompt: str, modello: str = "gpt-5", max_completion_tokens: int = 700) -> dict:
    # verifica l'ID modello aggiornato nella doc ufficiale
    risposta = cliente_openai.chat.completions.create(
        model=modello,
        max_completion_tokens=max_completion_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return {
        "provider": "openai",
        "modello": modello,
        "testo": risposta.choices[0].message.content,
        "token_input": risposta.usage.prompt_tokens,
        "token_output": risposta.usage.completion_tokens,
    }

print("chiama_claude e chiama_gpt pronte.")


---
## Cella 2 — Caso pratico 1: stesso prompt, due provider

Il prompt del piano alimentare della dispensa, identico su entrambi i
provider. Guarda in particolare: rispetto del budget, coerenza della
tabella su tutti e 7 i giorni, tono.

In [ ]:
PROMPT_PIANO_ALIMENTARE = """
Agisci come un nutrizionista esperto. Ricevi i dati di un cliente
delimitati dal tag <dati_cliente>. Genera un piano alimentare e di
esercizio per 7 giorni in formato tabellare (colonne: giorno, colazione,
pranzo, cena, spuntino, esercizio, riposo, costo giornaliero stimato).

Prima di generare il piano, verifica:
- Se il budget settimanale indicato è sufficiente per una dieta
  bilanciata; se non lo è, dai priorità a ingredienti economici.
- Se ci sono restrizioni alimentari; se presenti, escludi
  tassativamente gli alimenti incompatibili.
- Se il tempo disponibile per l'esercizio è coerente con la routine
  proposta.

Non superare mai il budget settimanale indicato. Tono motivante ma senza
tecnicismi.

<dati_cliente>
Budget settimanale: 60 euro
Restrizioni: senza glutine
Obiettivo: perdita di peso graduale
Tempo disponibile per esercizio: 30 minuti al giorno, 5 giorni a settimana
</dati_cliente>
"""

risultato_claude = chiama_claude(PROMPT_PIANO_ALIMENTARE)
risultato_gpt = chiama_gpt(PROMPT_PIANO_ALIMENTARE)

print(f"=== {risultato_claude['provider'].upper()} ({risultato_claude['modello']}) ===")
print(risultato_claude["testo"])
print(f"\n[token: {risultato_claude['token_input']} input / {risultato_claude['token_output']} output]")

print("\n" + "=" * 70 + "\n")

print(f"=== {risultato_gpt['provider'].upper()} ({risultato_gpt['modello']}) ===")
print(risultato_gpt["testo"])
print(f"\n[token: {risultato_gpt['token_input']} input / {risultato_gpt['token_output']} output]")


**Le tue osservazioni** (completa dopo aver letto entrambi gli output):

| Aspetto | Claude | GPT |
|---|---|---|
| Rispetta il budget di 60 euro? | ... | ... |
| La tabella è coerente su tutti i 7 giorni? | ... | ... |
| Tono percepito | ... | ... |
| Quale useresti per questo caso d'uso? | ... | ... |

---
## Cella 3 — Ragionamento forzato: prima e dopo, su entrambi i provider

Riusiamo il pattern Chain of Thought del Modulo 03 per simulare l'effetto
di una modalità di ragionamento più estesa, e osserviamo se cambia il
comportamento sui due provider allo stesso modo.

In [ ]:
problema_ragionamento = """
Un magazzino riceve 3 forniture al mese: la prima di 1200 unità a 2,10 euro
l'una, la seconda di 800 unità a 1,95 euro l'una, la terza di 1500 unità a
2,30 euro l'una. Il 4% delle unità totali risulta difettoso e va scartato.
Qual è il costo medio per unità NON difettosa ricevuta nel mese?
"""

prompt_diretto = f"{problema_ragionamento}\nRispondi solo con la cifra finale."

prompt_cot = f"""
{problema_ragionamento}

Risolvi passo per passo:
1. Calcola il costo totale di ciascuna fornitura.
2. Somma per ottenere il costo totale e le unità totali.
3. Calcola quante unità sono difettose e quante restano valide.
4. Dividi il costo totale per le unità valide.
5. Presenta il risultato finale etichettato chiaramente.
"""

for nome, prompt in [("DIRETTO", prompt_diretto), ("CHAIN OF THOUGHT", prompt_cot)]:
    print(f"--- {nome} / Claude ---")
    print(chiama_claude(prompt, max_tokens=500)["testo"])
    print(f"\n--- {nome} / GPT ---")
    print(chiama_gpt(prompt, max_completion_tokens=500)["testo"])
    print("=" * 70)


**Domanda per te:** entrambi i provider migliorano allo stesso modo con
il Chain of Thought, o noti differenze? Verifica tu stesso il calcolo a
mano (costo totale = 1200×2,10 + 800×1,95 + 1500×2,30; unità totali =
1200+800+1500; unità valide = unità totali × 0,96) e controlla quale
versione, su quale provider, arriva al risultato corretto in modo più
affidabile.

---
## Esercizio facoltativo — Aggiungi Gemini al confronto

**TODO (facoltativo, richiede `GOOGLE_API_KEY` e il pacchetto
`google-genai`):** scrivi una terza funzione `chiama_gemini` seguendo lo
stesso schema di `chiama_claude` e `chiama_gpt`, e ripeti il confronto
della Cella 2 con tutti e tre i provider.

In [ ]:
# pip install google-genai
# from google import genai
#
# cliente_gemini = genai.Client(api_key=GOOGLE_KEY)
#
# def chiama_gemini(prompt: str, modello: str = "gemini-3-flash") -> dict:
#     # verifica l'ID modello aggiornato nella doc ufficiale
#     risposta = cliente_gemini.models.generate_content(model=modello, contents=prompt)
#     return {
#         "provider": "google",
#         "modello": modello,
#         "testo": risposta.text,
#     }
#
# risultato_gemini = chiama_gemini(PROMPT_PIANO_ALIMENTARE)
# print(risultato_gemini["testo"])


---
## Sfida finale — Confronto cloud vs locale (concettuale)

Se hai installato [Ollama](https://ollama.com) o [LM Studio](https://lmstudio.ai)
sul tuo computer, prova questa sfida davvero: se non li hai installati, va
comunque bene rispondere alle domande in modo concettuale, ragionando su
quanto visto nella dispensa.

1. Scarica un modello open-weight di dimensioni contenute (per esempio una
   variante piccola di Llama o Mistral, qualche GB).
2. Sottoponigli lo stesso `PROMPT_PIANO_ALIMENTARE` della Cella 2.
3. Confronta il risultato con quelli di Claude e GPT ottenuti sopra.

Rispondi qui, anche solo concettualmente se non hai eseguito il test:

- Il modello locale rispetta tutti i vincoli del prompt (budget,
  restrizioni, formato tabellare) con la stessa affidabilità dei modelli
  cloud? ...
- Quanto tempo impiega a rispondere, rispetto alle chiamate API? ...
- In quale scenario reale (tuo o ipotetico) preferiresti comunque il
  modello locale, anche sapendo che la qualità potrebbe essere inferiore? ...

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 04 — Modulo 04 del Corso Completo di IA Generativa*